# Sequential but misaligned accesses

This notebook adapts `simpleExampleNonAligned.cu` and compares the same integer vector addition at offsets 0 and 1. The one-element offset is four bytes: it is correctly aligned for `int`, but not aligned to a 32-byte memory sector. The notebook validates each result against the CPU and reports average kernel time.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell shows the GPU and CUDA compiler available to the notebook.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 15:11:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

`CUDA_CHECK` handles CUDA errors independently of `TimingGPU`. The timer uses CUDA events and reports elapsed milliseconds for GPU work.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}
TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}
void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}
float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## CUDA kernel and host program

The allocation base from `cudaMalloc` is aligned. Passing a pointer shifted by one `int` preserves the language-level alignment required by the type, while changing which memory sectors a warp touches. On current NVIDIA GPUs, a full warp reading 32 adjacent four-byte values uses four 32-byte sectors when aligned and spans five when shifted by four bytes. Caches and reuse between neighboring warps can reduce the timing impact.

In [5]:
%%writefile code/chapter02_misaligned_access.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cstdio>
#include <cstdlib>

const int N = 1 << 20;
const int THREADS = 256;
const int REPEATS = 100;

__global__ void add_int(const int *a, const int *b, int *c, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

void launch_add(int offset, const int *a, const int *b, int *c, int n) {
    const int blocks = (n + THREADS - 1) / THREADS;
    add_int<<<blocks, THREADS>>>(a + offset, b + offset, c + offset, n);
    CUDA_CHECK(cudaGetLastError());
}

int main() {
    const size_t bytes = size_t(N + 1) * sizeof(int);
    const size_t vector_bytes = size_t(N) * sizeof(int);
    int *h_a = (int*)std::malloc(bytes);
    int *h_b = (int*)std::malloc(bytes);
    int *h_aligned = (int*)std::malloc(bytes);
    int *h_offset = (int*)std::malloc(bytes);
    int *d_a = nullptr, *d_b = nullptr, *d_aligned = nullptr, *d_offset = nullptr;
    if (!h_a || !h_b || !h_aligned || !h_offset) {
        std::fprintf(stderr, "Host allocation failed.\n");
        return EXIT_FAILURE;
    }
    for (int i = 0; i <= N; ++i) {
        h_a[i] = i % 127;
        h_b[i] = (i * 7) % 131;
    }
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_aligned, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_offset, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    launch_add(0, d_a, d_b, d_aligned, N);
    launch_add(1, d_a, d_b, d_offset, N);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_aligned, d_aligned, vector_bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_offset + 1, d_offset + 1, vector_bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < N; ++i) {
        if (h_aligned[i] != h_a[i] + h_b[i] ||
            h_offset[i + 1] != h_a[i + 1] + h_b[i + 1]) {
            std::fprintf(stderr, "CPU/GPU mismatch at i=%d.\n", i);
            return EXIT_FAILURE;
        }
    }
    std::printf("CPU vs aligned GPU: PASS (%d values)\n", N);
    std::printf("CPU vs one-int-offset GPU: PASS (%d values)\n", N);

    TimingGPU timer;
    float elapsed[2];
    for (int method = 0; method < 2; ++method) {
        const int offset = method;
        int *d_out = offset ? d_offset : d_aligned;
        for (int k = 0; k < 10; ++k) launch_add(offset, d_a, d_b, d_out, N);
        CUDA_CHECK(cudaDeviceSynchronize());
        timer.StartCounter();
        for (int k = 0; k < REPEATS; ++k) launch_add(offset, d_a, d_b, d_out, N);
        elapsed[method] = timer.GetCounter() / REPEATS;
    }
    std::printf("Aligned access:       %.6f ms/launch (average of %d)\n", elapsed[0], REPEATS);
    std::printf("One-int-offset access: %.6f ms/launch (average of %d)\n", elapsed[1], REPEATS);
    std::printf("Offset/aligned time ratio: %.3f\n", elapsed[1] / elapsed[0]);
    std::printf("The offset is 4 bytes: valid for int, but shifted relative to a 32-byte sector boundary.\n");

    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_aligned)); CUDA_CHECK(cudaFree(d_offset));
    std::free(h_a); std::free(h_b); std::free(h_aligned); std::free(h_offset);
    return EXIT_SUCCESS;
}


Writing code/chapter02_misaligned_access.cu


## Compile and run

The program checks the GPU result against a CPU reference and prints timings averaged over repeated kernel launches. The timing excludes allocation, host-device copies, and validation. Measurements depend on the selected GPU.

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_misaligned_access.cu code/TimingGPU.cu -o memory_example
./memory_example


CPU vs aligned GPU: PASS (1048576 values)
CPU vs one-int-offset GPU: PASS (1048576 values)
Aligned access:       0.051187 ms/launch (average of 100)
One-int-offset access: 0.054418 ms/launch (average of 100)
Offset/aligned time ratio: 1.063
The offset is 4 bytes: valid for int, but shifted relative to a 32-byte sector boundary.
